# xenafan.com: story-page enrichment

Notebook version of `fanfic-enrich-xenafan`. Fetches **part 1** of each story and adds the header block at the top of the page (disclaimer, warnings, sequel notes, song credits, author email) to the `titles.html` records.

- **Network:** the fetch cell requests every story page that has not been saved yet (and `titles.html`, if needed). Requests go one at a time, 5–10 s apart, with robots.txt honored; the run stops if robots.txt can't be read, or after 3 failures in a row.
- **Resumable:** saved pages are never downloaded again, so re-running after an interruption continues where it left off.
- **Sample first:** run with `RUN_ALL = False` (about 6 stories), review the headers, and only then set `RUN_ALL = True` for the full pass (about 1,450 requests, about 2.5 hours).

In [ ]:
import os
import pathlib
import sys

# Find the repository root (the folder holding pyproject.toml) whether this
# notebook was opened from notebooks/ or from the root, and work from there,
# so saved pages and outputs always go to <repo>/data/ (or FANFIC_DATA_DIR).
REPO_ROOT = pathlib.Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Open this notebook from inside the repository folder.")
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))  # works without `pip install -e .`
REPO_ROOT

## Settings

`CONTACT_EMAIL` is sent with every request (in the User-Agent) so site operators can reach whoever is crawling. Use your own address. Leave it blank to be asked the first time a download is needed, or set the `FANFIC_CONTACT_EMAIL` environment variable instead.

In [ ]:
CONTACT_EMAIL = ""  # your email address
RUN_ALL = False      # False = sample only; True = every story (~1,450 requests)
SAMPLE_SIZE = 6      # the sample covers one story per content directory first

In [ ]:
import dataclasses
import json

from fanfic_scraper.discover.xenafan import discover_xenafan
from fanfic_scraper.fetch import XENAFAN, FetchError, build_default_fetcher, default_data_dir
from fanfic_scraper.fetch.urlnorm import normalize_url
from fanfic_scraper.parse.xenafan import parse_xenafan_titles
from fanfic_scraper.parse.xenafan_story import apply_story_header, parse_xenafan_story
from fanfic_scraper.scripts.enrich_xenafan_stories import (
    MAX_CONSECUTIVE_FAILURES, part1_url, pick_sample,
)

fetcher = build_default_fetcher(XENAFAN, contact_email=CONTACT_EMAIL or None)
records = parse_xenafan_titles(discover_xenafan(fetcher).body)
all_targets = [(r, u) for r in records if (u := part1_url(r))]
targets = all_targets if RUN_ALL else pick_sample(all_targets, SAMPLE_SIZE)

skipped = len(records) - len(all_targets)
print(f"{len(records)} records in index; {skipped} skipped (external or lost); {len(targets)} to enrich")

## Fetch phase (network)

Fills the cache only. Stops after 3 failures in a row; re-run this cell later to resume.

In [ ]:
consecutive_failures = 0
for i, (_record, url) in enumerate(targets, 1):
    if fetcher.get_cached(url) is not None:
        continue
    try:
        fetcher.fetch(url)
        consecutive_failures = 0
        print(f"[{i}/{len(targets)}] fetched {url}")
    except FetchError as exc:
        consecutive_failures += 1
        print(f"[{i}/{len(targets)}] FAILED {url}: {exc}")
        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            print(f"Stopping: {consecutive_failures} failures in a row from {XENAFAN.host}.")
            break

cached_count = sum(fetcher.get_cached(u) is not None for _, u in targets)
print(f"{cached_count} of {len(targets)} story pages are saved")

## Parse phase (saved pages only)

In [ ]:
enriched = []
headers = {}
for record, url in targets:
    cached = fetcher.get_cached(url)
    if cached is None or cached.status != 200:
        continue
    header = parse_xenafan_story(cached.body, cached.url)
    apply_story_header(record, header, normalize_url(url, XENAFAN)[0])
    headers[record.source_id] = header
    enriched.append(record)

print(f"Enriched {len(enriched)} of {len(targets)} stories")

## Review the headers

Check that each header stops where the story text begins, and that the nav parts are real part links (not, say, a "sequel to" link in the header).

In [ ]:
for record in enriched[:SAMPLE_SIZE if not RUN_ALL else 10]:
    header = headers[record.source_id]
    print(f"== {record.title} by {', '.join(record.authors)}")
    print(f"   {header.url}")
    print(record.disclaimer_text or "(no header found)")
    print(f"-- meta author: {header.meta_author!r}   emails: {header.author_emails}")
    print(f"-- nav parts: {[p.label for p in header.nav_parts]}")
    print(f"-- author listing: {header.author_listing_url}")
    if record.notes:
        print(f"-- notes: {record.notes}")
    print()

## Look up one story

Edit the query and re-run to see the full enriched record.

In [ ]:
query = "Ab Initio"
for r in enriched:
    if query.lower() in r.title.lower():
        print(json.dumps(dataclasses.asdict(r), indent=2, ensure_ascii=False))

## Problems to check

Pages with no `<hr>` (header not extracted), pages whose meta author disagrees with the index, and part-count mismatches.

In [ ]:
no_header = [r for r in enriched if r.disclaimer_text is None]
mismatched = [r for r in enriched if r.notes and ("meta author" in r.notes or "nav lists" in r.notes)]
print(f"{len(no_header)} with no header found")
for r in no_header[:20]:
    print(f"    {r.title!r} by {r.authors}: {headers[r.source_id].url}")
print(f"\n{len(mismatched)} with author or part-count notes")
for r in mismatched[:20]:
    print(f"    {r.title!r} by {r.authors}: {r.notes}")

## Save

In [ ]:
out_records = records if RUN_ALL else enriched
out_path = default_data_dir() / ("xenafan_enriched.json" if RUN_ALL else "xenafan_enriched_sample.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump([dataclasses.asdict(r) for r in out_records], f, indent=2, ensure_ascii=False)
print(f"Wrote {len(out_records)} records to {out_path}")